# Exam Score Prediction - Improved Approach

This notebook implements advanced Data Science best practices to improve model performance. 
Key improvements over the baseline:
1. **Feature Engineering**: Creating interaction terms and using Ordinal Encoding for ranked categories.
2. **Outlier Removal**: Cleaning the training data to remove noise.
3. **Robust Preprocessing**: Manually fitting Scalers and Encoders on Training data only to prevent leakage.
4. **Ensemble Modeling**: Using a Voting Regressor to combine the strengths of multiple models.
5. **Strict Evaluation**: Only using the Test Set for the final check.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("Exam_Score_Prediction.csv")
df = df.drop("student_id", axis=1)
df.head()

## 1. Feature Engineering
We apply transformations *before* splitting where they are row-independent (like mapping values or mathematical operations on a single row). This makes the logic cleaner.

In [ ]:
# Best Practice: Accessing ordinal nature of data (Rank) rather than treating them as just 'categories'.
# This helps tree-based models make better splits (e.g. "High > Low").

quality_map = {'poor': 1, 'average': 2, 'good': 3}
difficulty_map = {'easy': 1, 'moderate': 2, 'hard': 3}
rating_map = {'low': 1, 'medium': 2, 'high': 3}

df['sleep_quality_ord'] = df['sleep_quality'].map(quality_map)
df['exam_difficulty_ord'] = df['exam_difficulty'].map(difficulty_map)
df['facility_rating_ord'] = df['facility_rating'].map(rating_map)

# Drop original columns to avoid multi-collinearity with the new ones
df = df.drop(['sleep_quality', 'exam_difficulty', 'facility_rating'], axis=1)

# Domain Knowledge: Study Hours *Attendance is "Effective Effort". 
# 10 hours study with 10% attendance is worse than 10 hours with 100% attendance.
df['effective_effort'] = df['study_hours'] * df['class_attendance']

# 'study_hours' often has diminishing returns. Log transform helps linear models.
df['log_study_hours'] = np.log1p(df['study_hours'])

print("New Feature shape:", df.shape)
df.head()

## 2. Data Splitting
Strict 70% Train, 15% Validation, 15% Test split.

In [ ]:
X = df.drop("exam_score", axis=1)
y = df["exam_score"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 3. Outlier Removal (Training Set Only)
We remove confusing data points from the Training set so the model isn't confused. We do NOT remove them from Val/Test, because we need to measure how the model handles real-world messiness.

In [ ]:
outlier_mask = (X_train['study_hours'] > 8) & (y_train < 30)
print(f"Removing {outlier_mask.sum()} outliers from training data.")

X_train = X_train[~outlier_mask]
y_train = y_train[~outlier_mask]

## 4. Manual Preprocessing (No Pipeline)
We will explicitly fit scalers on X_train and transform X_val/X_test.

In [ ]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

In [ ]:
scaler = StandardScaler()

scaler.fit(X_train[num_cols])

X_train_scaled_num = scaler.transform(X_train[num_cols])
X_val_scaled_num = scaler.transform(X_val[num_cols])
X_test_scaled_num = scaler.transform(X_test[num_cols])
X_train_num_df = pd.DataFrame(X_train_scaled_num, columns=num_cols, index=X_train.index)
X_val_num_df = pd.DataFrame(X_val_scaled_num, columns=num_cols, index=X_val.index)
X_test_num_df = pd.DataFrame(X_test_scaled_num, columns=num_cols, index=X_test.index)

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

encoder.fit(X_train[cat_cols])
X_train_enc = encoder.transform(X_train[cat_cols])
X_val_enc = encoder.transform(X_val[cat_cols])
X_test_enc = encoder.transform(X_test[cat_cols])

enc_cols = encoder.get_feature_names_out(cat_cols)
X_train_cat_df = pd.DataFrame(X_train_enc, columns=enc_cols, index=X_train.index)
X_val_cat_df = pd.DataFrame(X_val_enc, columns=enc_cols, index=X_val.index)
X_test_cat_df = pd.DataFrame(X_test_enc, columns=enc_cols, index=X_test.index)

X_train_final = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_val_final = pd.concat([X_val_num_df, X_val_cat_df], axis=1)
X_test_final = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

X_train_final.head()

## 5. Model Training & Tuning

In [ ]:
def evaluate_model(model, X, y, name="Model"):
    pred = model.predict(X)
    r2 = r2_score(y, pred)
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    print(f"--- {name} Results ---")
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    return r2

In [ ]:
ridge = Ridge()
params_ridge = {'alpha': [0.1, 1, 10, 50, 100]}
grid_ridge = GridSearchCV(ridge, params_ridge, cv=5, scoring='r2', n_jobs=-1)

grid_ridge.fit(X_train_final, y_train)
evaluate_model(grid_ridge.best_estimator_, X_val_final, y_val, "Ridge Tuned")
best_ridge = grid_ridge.best_estimator_

In [ ]:
rf = RandomForestRegressor(random_state=42)
params_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [2, 4]
}
grid_rf = GridSearchCV(rf, params_rf, cv=3, scoring='r2', n_jobs=-1)
grid_rf.fit(X_train_final, y_train)

evaluate_model(grid_rf.best_estimator_, X_val_final, y_val, "Random Forest Tuned")
best_rf = grid_rf.best_estimator_

In [ ]:
xgb = XGBRegressor(random_state=42, verbosity=0)
params_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb, params_xgb, cv=3, scoring='r2', n_jobs=-1)
grid_xgb.fit(X_train_final, y_train)

evaluate_model(grid_xgb.best_estimator_, X_val_final, y_val, "XGBoost Tuned")
best_xgb = grid_xgb.best_estimator_

## 6. Voting Regressor (Ensemble)
Combining all models to smooth out individual errors.

In [ ]:
voting_reg = VotingRegressor(
    estimators=[
        ('ridge', best_ridge),
        ('rf', best_rf),
        ('xgb', best_xgb)
    ]
)

voting_reg.fit(X_train_final, y_train)
print("Training complete.")

evaluate_model(voting_reg, X_val_final, y_val, "Voting Regressor (Validation)")

## 7. Final Test Evaluation
This is the **ONLY** time we touch the test set.

In [ ]:
final_preds = voting_reg.predict(X_test_final)

r2_final = r2_score(y_test, final_preds)
mae_final = mean_absolute_error(y_test, final_preds)
rmse_final = np.sqrt(mean_squared_error(y_test, final_preds))

print("="*30)
print("FINAL TEST SET PERFORMANCE")
print("="*30)
print(f"R2 Score: {r2_final:.4f}")
print(f"MAE: {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")

plt.figure(figsize=(10,6))
residuals = y_test - final_preds
sns.scatterplot(x=final_preds, y=residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel("Predicted Score")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residual Plot: Checking for Patterns in Errors")
plt.show()